# 🚀 RAG Workshop - Environment Setup

Welcome to the **RAG & Multimodal Knowledge Workshop**! This notebook will guide you through setting up all required Azure resources.

## What This Notebook Does

1. **Checks prerequisites** - Python version, required tools
2. **Guides Azure deployment** - Two options: automated or manual
3. **Generates your `.env` file** - All connection strings in one place
4. **Validates your setup** - Quick connectivity tests

## Prerequisites

Before starting, ensure you have:
- ✅ An Azure subscription with **Owner** or **Contributor** role
- ✅ Python 3.11+ installed (but < 3.14)
- ✅ Azure CLI installed (`az --version`)

---

## 🐍 Selecting a Python Environment (VS Code)

When you first open this notebook, VS Code will ask you to **select a Python environment**.

**Choose Python 3.11.x or 3.12.x** - This is required because:
- GraphRAG requires Python ≥3.11 and <3.14
- Some Azure SDKs work best with these versions

**How to select:**
1. Click "Select Kernel" in the top-right of the notebook
2. Choose "Python Environments"
3. Select a Python 3.11.x or 3.12.x version (e.g., `Python 3.12.12`)

> ⚠️ **Avoid**: Python 3.14+ (too new) or Python 3.10 and below (too old)

---

**Estimated time**: ~20 minutes

## Step 1: Check Prerequisites

Let's verify your environment is ready for the workshop.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

print("🔍 Checking Prerequisites...")
print("=" * 50)

# Check Python version
python_version = sys.version_info
python_ok = python_version >= (3, 11) and python_version < (3, 14)
print(f"\n📌 Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
if python_ok:
    print("   ✅ Python version is compatible (≥3.11, <3.14)")
else:
    print("   ❌ Python version must be ≥3.11 and <3.14")
    print("   💡 Install Python 3.11 or 3.12: brew install python@3.11 (macOS)")

# Check Azure CLI
az_available = shutil.which("az") is not None
print(f"\n📌 Azure CLI: {'Found' if az_available else 'Not found'}")
if az_available:
    try:
        result = subprocess.run(["az", "--version"], capture_output=True, text=True)
        az_version_line = result.stdout.split("\n")[0]
        print(f"   ✅ {az_version_line}")
    except Exception:
        print("   ✅ Azure CLI is available")
else:
    print("   ❌ Azure CLI not found")
    print("   💡 Install: https://aka.ms/installazurecli")

# Check jq (needed for deployment script)
jq_available = shutil.which("jq") is not None
print(f"\n📌 jq (JSON processor): {'Found' if jq_available else 'Not found'}")
if jq_available:
    print("   ✅ jq is available")
else:
    print("   ⚠️ jq not found (optional, used by deploy script)")
    print("   💡 Install: brew install jq (macOS) or apt install jq (Linux)")

# Check poppler (for PDF processing)
pdftoppm_available = shutil.which("pdftoppm") is not None
print(f"\n📌 Poppler (PDF tools): {'Found' if pdftoppm_available else 'Not found'}")
if pdftoppm_available:
    print("   ✅ Poppler is available")
else:
    print("   ⚠️ Poppler not found (needed for PDF figure extraction in later modules)")
    print("   💡 Install: brew install poppler (macOS) or apt install poppler-utils (Linux)")

# Summary
print("\n" + "=" * 50)
all_ok = python_ok and az_available
if all_ok:
    print("🎉 All required prerequisites are met!")
    print("\n➡️ Proceed to Step 2: Azure Setup")
else:
    print("⚠️ Some prerequisites are missing. Please install them before continuing.")

## Step 2: Azure Login & Subscription

Let's ensure you're logged into Azure and using the correct subscription.

In [ ]:
import subprocess
import json

print("🔐 Checking Azure Login Status...")
print("=" * 50)

try:
    # Check if logged in
    result = subprocess.run(
        ["az", "account", "show"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        account = json.loads(result.stdout)
        print(f"\n✅ Logged in as: {account.get('user', {}).get('name', 'Unknown')}")
        print(f"📌 Current Subscription: {account.get('name', 'Unknown')}")
        print(f"   ID: {account.get('id', 'Unknown')}")
        print(f"   State: {account.get('state', 'Unknown')}")
        
        # Store for later use
        SUBSCRIPTION_ID = account.get('id')
        print(f"\n➡️ If this is correct, proceed to Step 3")
        print(f"   To change subscription, run: az account set --subscription <subscription-id>")
    else:
        print("\n❌ Not logged in to Azure")
        print("\n💡 Run the cell below to login:")
        
except FileNotFoundError:
    print("\n❌ Azure CLI not found. Please install it first.")

In [ ]:
# Run this cell ONLY if you need to login to Azure
# This will open a browser window for authentication

!az login

In [ ]:
# Run this cell to list all available subscriptions
# Then use the next cell to set the correct one

!az account list --output table

In [ ]:
# Uncomment and run this cell to change your active subscription
# Replace <subscription-id> with your actual subscription ID

# !az account set --subscription "<subscription-id>"

## Step 3: Deploy Azure Resources

Choose one of two deployment options:

### Option A: Automated Deployment (Recommended)
Uses our Bicep template to deploy all resources at once.

### Option B: Manual Configuration
If you already have Azure resources or prefer manual setup.

---

### Resources to be deployed:

| Resource | Purpose |
|----------|----------|
| Azure OpenAI | GPT-4o, GPT-4o-mini (Module 1+2), GPT-4.1, GPT-4.1-mini (Module 3+), text-embedding-3-large |
| Azure AI Search | Vector + semantic search |
| Azure AI Services | Document Intelligence + Content Understanding |
| Storage Account | Document and figure storage |

### Model Deployments Explained:

| Model | Deployment Name | Used By |
|-------|-----------------|---------|
| gpt-4o | `gpt-4o` | Module 1, 2 (Basic RAG, Doc Intelligence) |
| gpt-4o-mini | `gpt-4o-mini` | Module 1, 2 (Cost-efficient option) |
| **gpt-4.1** | `gpt-4.1` | Module 3+ (Content Understanding prebuilt analyzers) |
| **gpt-4.1-mini** | `gpt-4.1-mini` | Module 3+ (Content Understanding `prebuilt-documentSearch`) |
| text-embedding-3-large | `text-embedding-3-large` | All modules (Vector embeddings) |

> ⚠️ **Important**: Content Understanding (`prebuilt-documentSearch`) **requires** `gpt-4.1-mini` specifically.
> See: [Azure AI Content Understanding Samples](https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models)

**Region**: `swedencentral` (required for Content Understanding GA)

### Option A: Automated Deployment

Run the cells below to deploy all Azure resources using our Bicep template.

In [ ]:

# Configuration - Modify these if needed
RESOURCE_GROUP = "rg-rag-workshop"
LOCATION = "swedencentral"  # Required for Content Understanding GA
# Changed name to 'ragworkv2' (no hyphens) to be storage-account compatible
BASE_NAME = "ragworkv2" 

print("📋 Deployment Configuration")
print("=" * 50)
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Location: {LOCATION}")
print(f"Base Name: {BASE_NAME}")
print("\n⚠️ Review the configuration above before proceeding.")
print("   Modify the variables in this cell if needed.")
# Note: Deployment uses GPT-4o instead of GPT-4.1 due to region compatibility


In [ ]:

import sys
import subprocess
import json
import time
from pathlib import Path

# Install azure-mgmt-resource if needed
try:
    from azure.mgmt.resource import ResourceManagementClient
except ImportError:
    print("Installing azure-mgmt-resource...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "azure-mgmt-resource", "azure-identity"])
    from azure.mgmt.resource import ResourceManagementClient

from azure.identity import AzureCliCredential
from azure.core.exceptions import HttpResponseError

# Path to Bicep file
bicep_path = Path("../../infra/main.bicep").resolve()
json_path = bicep_path.with_suffix(".json")

print(f"🚀 Starting deployment for {BASE_NAME} in {LOCATION} via Python SDK...")
print(f"   using Bicep template: {bicep_path}")

# Step 0: Build Bicep to JSON (SDK requires JSON)
print(f"\n🔨 Compiling Bicep to JSON...")
build_result = subprocess.run(
    ["az", "bicep", "build", "--file", str(bicep_path), "--outfile", str(json_path)],
    capture_output=True,
    text=True
)
if build_result.returncode != 0:
    print(f"❌ Bicep build failed: {build_result.stderr}")
    raise Exception(f"Bicep build failed: {build_result.stderr}")

with open(json_path, 'r') as f:
    template_body = json.load(f)

# Step 1: Create Resource Group
print(f"\n📦 Creating Resource Group: {RESOURCE_GROUP}...")
try:
    credential = AzureCliCredential()
    resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)

    rg_params = {"location": LOCATION, "tags": {"workshop": "rag-workshop"}}
    resource_client.resource_groups.create_or_update(RESOURCE_GROUP, rg_params)
    print("   ✅ Resource group created/confirmed")
except Exception as e:
    print(f"   ❌ Failed to create resource group: {str(e)}")
    print("   Ensure you are logged in using 'az login'")
    raise

# Step 2: Deploy
print(f"\n🔧 Deploying Azure resources (SDK)...")
print("   This may take several minutes...")

deployment_properties = {
    "mode": "Incremental",
    "template": template_body,
    "parameters": {
        "baseName": {"value": BASE_NAME},
        "location": {"value": LOCATION}
    }
}

try:
    deployment_async_operation = resource_client.deployments.begin_create_or_update(
        RESOURCE_GROUP,
        "main",
        {"properties": deployment_properties}
    )
    
    # Wait for completion
    deployment_result = deployment_async_operation.result()
    print("   ✅ Deployment completed successfully!")
    
    # Extract outputs
    deployment_outputs = {}
    if deployment_result.properties.outputs:
        for key, value in deployment_result.properties.outputs.items():
            deployment_outputs[key] = value # Keep structure or flatten? 
            # Previous usage expects: deployment_outputs[key]['value']
            # SDK returns dict where values are dicts with 'type' and 'value'.
            # So deployment_outputs is already in the right format.
            deployment_outputs = deployment_result.properties.outputs
            
        print("\n📋 Deployed Resources:")
        for key in deployment_outputs:
            val = deployment_outputs[key].get('value')
            if 'Endpoint' in key or 'Key' in key or 'Name' in key:
                print(f"   • {key}: {val}")

except HttpResponseError as e:
    print(f"   ❌ Deployment failed!")
    print(f"   Error: {e}")
    raise

except Exception as e:
    print(f"   ❌ An unexpected error occurred: {e}")
    raise


In [ ]:

# Generate .env file from deployment outputs
from pathlib import Path
from datetime import datetime
import subprocess
import json

print("📝 Generating .env file...")
print("=" * 50)

# Get subscription ID
sub_result = subprocess.run(
    ["az", "account", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True
)
subscription_id = sub_result.stdout.strip()

# Check if deployment_outputs exists from previous cell
try:
    outputs = deployment_outputs
except NameError:
    # Re-fetch deployment outputs
    print("   Fetching deployment outputs...")
    result = subprocess.run(
        [
            "az", "deployment", "group", "show",
            "--resource-group", RESOURCE_GROUP,
            "--name", "main",
            "--query", "properties.outputs",
            "--output", "json"
        ],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        outputs = json.loads(result.stdout)
    else:
        print(f"❌ Could not fetch deployment outputs: {result.stderr}")
        raise Exception("Run the deployment cell first")

# Helper to safely get output values
def get_output(key, default=""):
    return outputs.get(key, {}).get('value', default)

# Unified AI Services Endpoint & Key
ai_endpoint = get_output('aiServicesEndpoint')
ai_key = get_output('aiServicesKey')

# Extract values
env_content = f'''# ===========================================
# RAG Workshop Environment Configuration
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Region: {LOCATION}
# ===========================================

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RESOURCE_GROUP}
AZURE_LOCATION={LOCATION}

# Unified Azure AI Services (The Engine)
# Handles OpenAI, Document Intelligence, Content Understanding
AZURE_AI_SERVICES_ENDPOINT={ai_endpoint}
AZURE_AI_SERVICES_KEY={ai_key}

# Azure OpenAI (Uses Unified Endpoint)
AZURE_OPENAI_ENDPOINT={ai_endpoint}
AZURE_OPENAI_API_KEY={ai_key}
AZURE_OPENAI_API_VERSION=2024-08-01-preview

# --- Model Deployments for Module 1+2 (GPT-4o) ---
AZURE_OPENAI_DEPLOYMENT_GPT4o=gpt-4o
AZURE_OPENAI_DEPLOYMENT_GPT4o_MINI=gpt-4o-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# --- Model Deployments for Module 3+ (Content Understanding requires GPT-4.1) ---
# See: https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models
AZURE_OPENAI_DEPLOYMENT_GPT41=gpt-4.1
AZURE_OPENAI_DEPLOYMENT_GPT41_MINI=gpt-4.1-mini
GPT_4_1_DEPLOYMENT=gpt-4.1
GPT_4_1_MINI_DEPLOYMENT=gpt-4.1-mini
TEXT_EMBEDDING_3_LARGE_DEPLOYMENT=text-embedding-3-large

# --- Model Deployment for Module 5 Agentic Retrieval ---
# gpt-4.1 supports Structured Outputs required by Agentic Retrieval
AZURE_OPENAI_DEPLOYMENT_AGENTIC=gpt-4.1

# Azure AI Search
AZURE_SEARCH_ENDPOINT={get_output('searchServiceEndpoint')}
AZURE_SEARCH_API_KEY={get_output('searchServiceAdminKey')}
AZURE_SEARCH_INDEX_NAME=rag-workshop-index

# Azure AI Document Intelligence & Content Understanding (Uses Unified Endpoint)
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={ai_endpoint}
AZURE_DOCUMENT_INTELLIGENCE_KEY={ai_key}
AZURE_CONTENT_UNDERSTANDING_ENDPOINT={ai_endpoint}
AZURE_CONTENT_UNDERSTANDING_KEY={ai_key}
AZURE_CONTENT_UNDERSTANDING_API_VERSION=2025-11-01

# Azure Storage
AZURE_STORAGE_CONNECTION_STRING={get_output('storageAccountConnectionString')}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
AZURE_STORAGE_CONTAINER_FIGURES=figures

# GraphRAG (uses Azure OpenAI settings)
GRAPHRAG_API_KEY=${{AZURE_OPENAI_API_KEY}}
GRAPHRAG_API_BASE=${{AZURE_OPENAI_ENDPOINT}}
GRAPHRAG_API_VERSION=${{AZURE_OPENAI_API_VERSION}}
'''

# Write .env file to project root
env_path = Path("../../.env").resolve()
env_path.write_text(env_content)

print(f"\n✅ .env file created at: {env_path}")
print("\n📋 Model Deployments:")

print("   Module 1+2: gpt-4o, gpt-4o-mini (standard GPT-4o)")

print("   Module 3+:  gpt-4.1, gpt-4.1-mini (Content Understanding)")print("\n➡️ Proceed to Step 4 or run health-check.ipynb to validate.")

print("\n⚠️ IMPORTANT: The .env file contains sensitive API keys.")print("   It is already in .gitignore - DO NOT commit it to git!")

### Option B: Manual Configuration

If you already have Azure resources or prefer manual setup, run the cell below to create your `.env` file interactively.

In [ ]:
# SKIP this cell if you used Option A (automated deployment)

from pathlib import Path
from datetime import datetime

print("📝 Manual Environment Configuration")
print("=" * 50)
print("\nEnter your Azure resource details below.")
print("Leave blank and press Enter to use the default value in brackets.\n")

def get_input(prompt, default=""):
    display = f"{prompt} [{default}]: " if default else f"{prompt}: "
    value = input(display).strip()
    return value if value else default

# Collect values
print("--- Azure Subscription ---")
subscription_id = get_input("Subscription ID")
resource_group = get_input("Resource Group", "rg-rag-workshop")
location = get_input("Location", "swedencentral")

print("\n--- Azure OpenAI ---")
openai_endpoint = get_input("OpenAI Endpoint (e.g., https://xxx.openai.azure.com/)")
openai_key = get_input("OpenAI API Key")

print("\n--- Azure AI Search ---")
search_endpoint = get_input("Search Endpoint (e.g., https://xxx.search.windows.net)")
search_key = get_input("Search Admin Key")

print("\n--- Azure AI Services (Document Intelligence) ---")
di_endpoint = get_input("AI Services Endpoint (e.g., https://xxx.cognitiveservices.azure.com/)")
di_key = get_input("AI Services Key")

print("\n--- Azure Storage ---")
storage_conn = get_input("Storage Connection String")

# Generate .env content
env_content = f'''# ===========================================
# RAG Workshop Environment Configuration
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Region: {location}
# ===========================================

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={resource_group}
AZURE_LOCATION={location}

# Azure OpenAI
AZURE_OPENAI_ENDPOINT={openai_endpoint}
AZURE_OPENAI_API_KEY={openai_key}
AZURE_OPENAI_API_VERSION=2024-08-01-preview

# --- Model Deployments for Module 1+2 (GPT-4o) ---
AZURE_OPENAI_DEPLOYMENT_GPT4o=gpt-4o
AZURE_OPENAI_DEPLOYMENT_GPT4o_MINI=gpt-4o-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# --- Model Deployments for Module 3+ (Content Understanding requires GPT-4.1) ---
# See: https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models
AZURE_OPENAI_DEPLOYMENT_GPT41=gpt-4.1
AZURE_OPENAI_DEPLOYMENT_GPT41_MINI=gpt-4.1-mini
GPT_4_1_DEPLOYMENT=gpt-4.1
GPT_4_1_MINI_DEPLOYMENT=gpt-4.1-mini
TEXT_EMBEDDING_3_LARGE_DEPLOYMENT=text-embedding-3-large

# --- Model Deployment for Module 5 Agentic Retrieval ---
# gpt-4.1 supports Structured Outputs required by Agentic Retrieval
AZURE_OPENAI_DEPLOYMENT_AGENTIC=gpt-4.1

# Azure AI Search
AZURE_SEARCH_ENDPOINT={search_endpoint}
AZURE_SEARCH_API_KEY={search_key}
AZURE_SEARCH_INDEX_NAME=rag-workshop-index

# Azure AI Document Intelligence & Content Understanding
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={di_endpoint}
AZURE_DOCUMENT_INTELLIGENCE_KEY={di_key}
AZURE_CONTENT_UNDERSTANDING_ENDPOINT={di_endpoint}
AZURE_CONTENT_UNDERSTANDING_KEY={di_key}
AZURE_CONTENT_UNDERSTANDING_API_VERSION=2025-11-01

# Azure Storage
AZURE_STORAGE_CONNECTION_STRING={storage_conn}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
AZURE_STORAGE_CONTAINER_FIGURES=figures

# GraphRAG (uses Azure OpenAI settings)
GRAPHRAG_API_KEY=${{AZURE_OPENAI_API_KEY}}
GRAPHRAG_API_BASE=${{AZURE_OPENAI_ENDPOINT}}
GRAPHRAG_API_VERSION=${{AZURE_OPENAI_API_VERSION}}
'''

# Write .env file
env_path = Path("../../.env").resolve()
env_path.write_text(env_content)

print(f"\n✅ .env file created at: {env_path}")

print("\n📋 Model Deployments:")print("\n➡️ Proceed to Step 4 to validate your setup.")

print("   Module 1+2: gpt-4o, gpt-4o-mini (standard GPT-4o)")print("   for Module 3 (Content Understanding) to work properly.")

print("   Module 3+:  gpt-4.1, gpt-4.1-mini (Content Understanding)")print("\n⚠️ NOTE: You must deploy gpt-4.1 and gpt-4.1-mini models in Azure AI Foundry")

## Step 4: Install Python Dependencies

Let's install the required Python packages for the workshop.

In [ ]:
# Install required packages
print("📦 Installing Python dependencies...")
print("=" * 50)
print("⏱️ This may take a few minutes.\n")

!pip install -q -r ../../requirements.txt

print("\n✅ Dependencies installed!")

## Step 5: Quick Validation

Let's do a quick test to ensure your environment is correctly configured.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

print("🔍 Loading and Validating Environment...")
print("=" * 50)

# Load .env file
env_path = Path("../../.env").resolve()
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"❌ .env file not found at: {env_path}")
    print("   Please run Step 3 first.")
    raise FileNotFoundError(".env file not found")

# Check required environment variables
required_vars = [
    ("AZURE_OPENAI_ENDPOINT", "Azure OpenAI"),
    ("AZURE_OPENAI_API_KEY", "Azure OpenAI"),
    ("AZURE_SEARCH_ENDPOINT", "Azure AI Search"),
    ("AZURE_SEARCH_API_KEY", "Azure AI Search"),
    ("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT", "Document Intelligence"),
    ("AZURE_DOCUMENT_INTELLIGENCE_KEY", "Document Intelligence"),
]

print("\n📋 Checking Environment Variables:")
all_present = True
for var_name, service in required_vars:
    value = os.getenv(var_name)
    if value:
        # Mask sensitive values
        if 'KEY' in var_name or 'SECRET' in var_name:
            display = value[:8] + "..." + value[-4:] if len(value) > 12 else "***"
        else:
            display = value[:50] + "..." if len(value) > 50 else value
        print(f"   ✅ {var_name}: {display}")
    else:
        print(f"   ❌ {var_name}: NOT SET ({service})")
        all_present = False

if all_present:
    print("\n🎉 All required environment variables are set!")
    print("\n➡️ Run health-check.ipynb for detailed connectivity tests.")
else:
    print("\n⚠️ Some environment variables are missing.")
    print("   Please check your .env file or re-run Step 3.")

In [ ]:

# Quick connectivity test - Azure OpenAI
# Fix for potential dependency conflicts
import sys
import subprocess
print("🔧 Ensuring compatible dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "typing_extensions", "pydantic", "pydantic-core"])

import os
from openai import AzureOpenAI

print("🔗 Testing Azure OpenAI Connection...")
print("=" * 50)

try:
    client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    
    # Test with a simple completion
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4o"),
        messages=[
            {"role": "user", "content": "Say 'Hello, RAG Workshop!' in exactly those words."}
        ],
        max_tokens=20
    )
    
    print(f"\n✅ Azure OpenAI (GPT-4o) is working!")
    print(f"   Response: {response.choices[0].message.content}")
    
except ImportError:
    print("\n⚠️ Dependency update required a kernel restart.")
    print("   Please restart the kernel and run this cell again.")
except Exception as e:
    print(f"\n❌ Azure OpenAI connection failed!")
    print(f"   Error: {str(e)}")
    print("\n💡 Troubleshooting:")
    print("   1. Verify AZURE_OPENAI_ENDPOINT is correct")
    print("   2. Check that gpt-4o deployment exists")
    print("   3. Ensure API key has access to the resource")


## 🎉 Setup Complete!

Congratulations! Your environment is ready for the RAG Workshop.

### Next Steps

1. **Run the full health check**: Open and run `health-check.ipynb` to validate all services
2. **Start Module 1**: Proceed to `../module-1-naive-rag/` to begin the workshop

### Quick Reference

| Resource | Status |
|----------|--------|
| Azure OpenAI | ✅ Configured |
| Azure AI Search | ✅ Configured |
| Document Intelligence | ✅ Configured |
| Content Understanding | ✅ Configured |
| Storage Account | ✅ Configured |

### Troubleshooting

If you encounter issues:
1. Run `health-check.ipynb` for detailed diagnostics
2. Check the Azure Portal for resource status
3. Verify your `.env` file has correct values

---

**[Next Module →](../module-1-naive-rag/README.md)** The Problem with Naive RAG